In [74]:
import pysynphot as S
import config
import pandas as pd
import os
import numpy as np
from pathlib import Path

In [15]:
df_spec = pd.read_csv(
    config.DATA_DIR_MISC / "spectra" / "pickle_spectra.txt", sep="\s+",
    index_col="FileName"
)

In [22]:
bandpass_root = config.DATA_DIR_MISC / "bandpasses"
bp_band = S.FileBandpass(str(bandpass_root / "gaia_dr3_bp.dat"))
rp_band = S.FileBandpass(str(bandpass_root / "gaia_dr3_rp.dat"))

In [90]:
bands = {
    "BP": bp_band, "RP":rp_band
}

obs_vega_bp = S.Observation(S.Vega, band=bp_band)
obs_vega_rp = S.Observation(S.Vega, band=rp_band)

vega_obs_dict = {
    "BP": obs_vega_bp,
    "RP": obs_vega_rp
}

spec_root = Path(os.environ["PYSYN_CDBS"]) / "grid" / "pickles" / "dat_uvk"

result_data = []

for file_stem, row in df_spec.iterrows():
    spec_file_path = spec_root / f"{file_stem}.fits"
    spec = S.FileSpectrum(str(spec_file_path))
    row_data = []
    for key, val in bands.items():
        obs = S.Observation(spec, band=val)
        obs_vega = vega_obs_dict[key]
        mag = (
            -2.5 * np.log10(
                obs_vega.sample(obs.bandpass.pivot())
                / obs.effstim(fluxunits="counts") 
            )
        )
        row_data.extend([mag])

    result_data.append(row_data)

mag_df = pd.DataFrame(
    columns=["BP", "RP"], index=df_spec.index, data=result_data
)

df_combined = df_spec.join(mag_df)
df_combined["bp_rp"] = df_combined["BP"] - df_combined["RP"]
df_combined.to_csv(config.DATA_DIR_MISC / "spec_type_and_bp_rp.csv", 
                   index=False)

(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_bp.dat) does not have a defined binset in the wavecat table. The waveset of the spectrum will be used instead.
(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_rp.dat) does not have a defined binset in the wavecat table. The waveset of the spectrum will be used instead.
(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_bp.dat) does not have a defined binset in the wavecat table. The waveset of the spectrum will be used instead.
(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_rp.dat) does not have a defined binset in the wavecat table. The waveset of the spectrum will be used instead.
(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_bp.dat) does not have a defined binset in the wavecat table. The w

In [106]:
spec_file_path = spec_root / "pickles_uk_1.fits"
spec_test = S.FileSpectrum(str(spec_file_path))
obs_test = S.Observation(spec=spec_test, band=rp_band)

(/Users/yuezhao/Desktop/local_repos/runaway_high_v_sources/data/combined/misc/bandpasses/gaia_dr3_rp.dat) does not have a defined binset in the wavecat table. The waveset of the spectrum will be used instead.


In [ ]:
# 2.5 * np.log10( / )

6.599744545981329

In [111]:
obs.effstim("photlam")

3432.3531165405634

In [114]:
photlam = S.units.Photlam()

In [117]:
photlam.ToVegaMag(wave=obs.pivot(), flux=obs.effstim())

TypeError: 'float' object is not subscriptable

In [118]:
obs.pivot()

8050.99947932994